# Lab | BabyAGI with agent

**Change the planner objective below by changing the objective and the associated prompts and potential tools and agents - Wear your creativity and AI engineering hats
You can't get this wrong!**

You would need the OpenAI API KEY and the [SerpAPI KEY](https://serpapi.com/manage-api-keyhttps://serpapi.com/manage-api-key) to run this lab.


## BabyAGI with Tools

This notebook builds on top of [baby agi](baby_agi.html), but shows how you can swap out the execution chain. The previous execution chain was just an LLM which made stuff up. By swapping it out with an agent that has access to tools, we can hopefully get real reliable information

## Install and Import Required Modules

In [2]:
!pip install -U langchain langchain-openai langchain-community



   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 93.7/93.7 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 82.5/82.5 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 45.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 35.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 473.0/473.0 kB 25.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 156.8/156.8 kB 10.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.2/46.2 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.8/56.8 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 208.3/208.3 kB 15.4 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully 

In [3]:
!pip install langgraph


In [4]:
from typing import Optional

from langchain_core.runnables import RunnableSequence
from langgraph.prebuilt import create_react_agent
from langchain_openai import OpenAI, OpenAIEmbeddings

## Connect to the Vector Store

Depending on what vectorstore you use, this step may look different.

In [5]:
# %pip install faiss-cpu > /dev/null
# %pip install google-search-results > /dev/null
from langchain_community.docstore.in_memory import InMemoryDocstore
from langchain_community.vectorstores import FAISS

In [59]:


OPENAI_API_KEY  = "OPENAI_API_KEY"
SERP_API_KEY = "SERP_KEY"

In [7]:
!pip install faiss-cpu


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.6/23.6 MB 81.7 MB/s eta 0:00:00


In [8]:
# Define your embedding model
embeddings_model = OpenAIEmbeddings(openai_api_key=OPENAI_API_KEY)
# Initialize the vectorstore as empty
import faiss

embedding_size = 1536
index = faiss.IndexFlatL2(embedding_size)
vectorstore = FAISS(embeddings_model.embed_query, index, InMemoryDocstore({}), {})
# Helper functions for memory
def add_to_memory(task: str, result: str):
    vectorstore.add_texts([result], metadatas=[{"task": task}])

def retrieve_context(query: str, k: int = 3) -> str:
    docs = vectorstore.similarity_search(query, k=k)
    return "\n".join([doc.page_content for doc in docs])

In [9]:
!pip install google-search-results

  Preparing metadata (setup.py) ... done
  Created wheel for google-search-results: filename=google_search_results-2.4.2-py3-none-any.whl size=32010 sha256=45fba5b6e6dbfa23ea43971bcdf1fcd89824cafbf847bbd7ee7c6362dc4adb58
  Stored in directory: /root/.cache/pip/wheels/0c/47/f5/89b7e770ab2996baf8c910e7353d6391e373075a0ac213519e
Successfully built google-search-results


## Define the Chains

BabyAGI relies on three LLM chains:
- Task creation chain to select new tasks to add to the list
- Task prioritization chain to re-prioritize tasks
- Execution Chain to execute the tasks


NOTE: in this notebook, the Execution chain will now be an agent.

In [51]:
from langchain.agents import create_agent
from langchain.tools import tool
from langchain_core.runnables import RunnableSequence
from langchain_core.prompts import PromptTemplate
from langchain_community.utilities import SerpAPIWrapper
from langchain_openai import ChatOpenAI
execution_prompt = PromptTemplate(
    input_variables=["objective", "task", "context"],
    template="""
    You are an AI research assistant. Given the following task,
    think step by step before answering. Show your reasoning
    first, then provide a concise final answer.

    Objective: {objective}
    Task: {task}
    Context: {context}

    Step-by-step reasoning:
    1.
    2.
    3.
    Final Answer:
    """
    )

task_creation_prompt = PromptTemplate(
    input_variables=["objective", "result", "task_list"],
    template="""
Based on the last result: {result}, create 1–5 new tasks that are specific, distinct, and advance the objective: {objective}.
Do not repeat tasks already in {task_list}.
Output tasks as a numbered list, one task per line.

"""
)

task_prioritization_prompt = PromptTemplate(
    input_variables=["objective", "task_list"],
    template="""
You are a task manager. Given the list of tasks:
{task_list}
Prioritize them in order of importance to achieve the objective: {objective}
Output a numbered list of tasks.

"""
)


In [53]:

llm=ChatOpenAI(temperature=0,openai_api_key=OPENAI_API_KEY)
# todo_chain = RunnableSequence([todo_prompt,llm])
execution_chain = RunnableSequence(first=execution_prompt,middle=[], last=llm)
task_creation_chain = RunnableSequence(first=task_creation_prompt,middle=[], last=llm)
task_prioritization_chain = RunnableSequence(first=task_prioritization_prompt,middle=[], last=llm)


def todo_tool(objective: str, last_result: str = "", task_list_text: str = "") -> str:
    """
    Generates new tasks based on objective, last result, and incomplete tasks.
    Returns tasks as string (one per line).
    """
    output = task_creation_chain.invoke({
        "objective": objective,
        "result": last_result,
        "task_list": task_list_text
    })
    # Convert AIMessage to string if necessary
    if hasattr(output, "content"):
        return output.content
    return output

search = SerpAPIWrapper(serpapi_api_key="SERP_API_KEY")

def search_tool(query: str) -> str:
    """
    Searches the web using SerpAPI and returns relevant results.
    """
    return search.run(query)



In [54]:
tools = [todo_tool, search_tool]

agent = create_agent(
    llm,
    tools=tools,
    system_prompt="You are an autonomous AI agent. Use tools wisely."
)

In [55]:
from collections import deque
from collections import deque

def normalize_output(output):
    if hasattr(output, "content"):
        return str(output.content)
    return str(output)

def babyagi_controller(objective: str, max_iterations: int = 5):
    task_list = deque(["Start initial research"])
    completed_tasks = []

    for _ in range(max_iterations):
        if not task_list:
            break

        task = task_list.popleft()

        # Retrieve relevant memory
        context = retrieve_context(task)

        # Execute task
        output = execution_chain.invoke({
            "objective": objective,
            "task": task,
            "context": context
        })
        result = normalize_output(output)
        completed_tasks.append(f"{task}: {result}")

        # Store result in vectorstore memory
        add_to_memory(task, result)

        # Generate new tasks
        new_tasks_text = todo_tool(
            objective,
            last_result=result,
            task_list_text="\n".join(task_list)
        )
        new_tasks_text = normalize_output(new_tasks_text)
        for t in new_tasks_text.split("\n"):
            t = t.strip()
            if t:
                # Remove numbering if present
                if ". " in t:
                    t = t.split(". ", 1)[1]
                task_list.append(t)

        # Re-prioritize tasks
        if task_list:
            prioritized_text = task_prioritization_chain.invoke({
                "objective": objective,
                "task_list": "\n".join(task_list)
            })
            prioritized_text = normalize_output(prioritized_text)

            # Update task_list safely
            task_list = deque([
                t.split(". ", 1)[1] if ". " in t else t
                for t in prioritized_text.split("\n")
                if t.strip()
            ])

    return completed_tasks


In [56]:
objective = "Build a list of AI startups in 2025"

completed = babyagi_controller(objective, max_iterations=5)

print("Completed Tasks:\n")
for task in completed:
    print(task)


Completed Tasks:

Start initial research: Step-by-step reasoning:
1. Conduct thorough research on the current landscape of AI startups.
2. Reach out to industry experts, venture capitalists, and stakeholders in the AI ecosystem for insights.
3. Attend AI conferences, networking events, and pitch competitions to discover new startups.

Final Answer:
Based on initial research, some of the top AI startups in 2025 include OpenAI, DeepMind, UiPath, DataRobot, Cerebras Systems, Scale AI, Graphcore, Vicarious, Suki.ai, and Landing AI.
Analyze the funding rounds and investments of the top AI startups to understand their growth trajectory.: To build a list of AI startups in 2025, I would start by conducting thorough research on the current landscape of AI startups. This would involve looking into industry reports, startup databases, news articles, and other sources to identify emerging companies in the field of artificial intelligence.

I would also reach out to industry experts, venture capita

### Run the BabyAGI

Now it's time to create the BabyAGI controller and watch it try to accomplish your objective.

In [57]:
OBJECTIVE = "Write a weather report for SF today"

In [40]:
# # Logging of LLMChains
# verbose = False
# # If None, will keep on going forever
# max_iterations: Optional[int] = 3
# baby_agi = BabyAGI.from_llm(
#     llm=llm,
#     vectorstore=vectorstore,
#     task_execution_chain=agent_executor,
#     verbose=verbose,
#     max_iterations=max_iterations,
# )

In [58]:
babyagi_controller(objective, max_iterations=5)

['Start initial research: To build a list of AI startups in 2025, I would conduct thorough research, reach out to industry experts, and attend relevant events to identify and compile a comprehensive list of innovative AI startups shaping the future of artificial intelligence in 2025.',
 'Analyze industry reports and market trends to pinpoint key players and disruptors in the AI startup landscape.: To build a list of AI startups in 2025, I would conduct thorough research on the current landscape of AI startups, reach out to industry experts for insights, and attend relevant events to discover new and innovative startups.',
 'Utilize online databases and platforms such as Crunchbase and AngelList to identify emerging AI startups in various sectors.: 1. Conduct thorough research on the current landscape of AI startups by looking into industry reports, startup databases, news articles, and other sources.\n2. Reach out to industry experts, venture capitalists, and other stakeholders in the 